In [11]:
from pathlib import Path

import pandas as pd
import numpy as np
import json
import tarfile

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 300)

PROJECT_DIR = Path("..").resolve()

PUBLIC_RAW = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "raw"
)

INTERIM_DIR = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "interim"
)

REPORTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "tables"
)

RICO_DIR = (
    PUBLIC_RAW
    / "10_uicrit_rico"
)

INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [12]:
uicrit = pd.read_csv(
    RICO_DIR
    / "uicrit_public.csv"
)

ui_details = pd.read_csv(
    RICO_DIR
    / "ui_details.csv"
)

print(
    "UICrit:",
    uicrit.shape
)

print(
    "UI details:",
    ui_details.shape
)

print(
    uicrit.columns.tolist()
)

print(
    ui_details.columns.tolist()
)

UICrit: (2981, 9)
UI details: (66261, 4)
['rico_id', 'task', 'aesthetics_rating', 'learnability', 'efficency', 'usability_rating', 'design_quality_rating', 'comments_source', 'comments']
['UI Number', 'App Package Name', 'Interaction Trace Number', 'UI Number in Trace']


In [13]:
uicrit["rico_id_num"] = pd.to_numeric(
    uicrit["rico_id"],
    errors="coerce"
).astype("Int64")

ui_details["rico_id_num"] = pd.to_numeric(
    ui_details["UI Number"],
    errors="coerce"
).astype("Int64")

In [14]:
target_rico_ids = set(
    uicrit[
        "rico_id_num"
    ]
    .dropna()
    .astype(int)
)

rico_available_ids = set(
    ui_details[
        "rico_id_num"
    ]
    .dropna()
    .astype(int)
)

overlap = (
    target_rico_ids
    &
    rico_available_ids
)

print(
    "UICrit IDs:",
    len(target_rico_ids)
)

print(
    "Encontrados en RICO:",
    len(overlap)
)

print(
    "Cobertura:",
    len(overlap)
    / len(target_rico_ids)
)

UICrit IDs: 1000
Encontrados en RICO: 1000
Cobertura: 1.0


In [15]:
archive_path = (
    RICO_DIR
    / "unique_uis.tar.gz"
)

print(
    "Archivo:",
    archive_path.name
)

print(
    "GB:",
    archive_path.stat().st_size
    / 1024**3
)

Archivo: unique_uis.tar.gz
GB: 6.026833130978048


In [16]:
selected_hierarchies = {}
selected_member_names = []

with tarfile.open(
    archive_path,
    mode="r:gz"
) as tar:

    for member in tar:

        if not member.isfile():
            continue

        path = Path(
            member.name
        )

        if (
            path.suffix.lower()
            != ".json"
        ):
            continue

        try:
            rico_id = int(
                path.stem
            )
        except ValueError:
            continue

        if rico_id not in target_rico_ids:
            continue

        extracted = tar.extractfile(
            member
        )

        if extracted is None:
            continue

        try:
            obj = json.load(
                extracted
            )
        except Exception as e:

            print(
                "Error JSON:",
                rico_id,
                e
            )

            continue

        selected_hierarchies[
            rico_id
        ] = obj

        selected_member_names.append(
            member.name
        )

        if (
            len(selected_hierarchies)
            ==
            len(target_rico_ids)
        ):
            break

In [17]:
print(
    "Jerarquías recuperadas:",
    len(selected_hierarchies)
)

print(
    "Esperadas:",
    len(target_rico_ids)
)

print(
    "Faltantes:",
    len(
        target_rico_ids
        -
        set(
            selected_hierarchies.keys()
        )
    )
)

print(
    selected_member_names[:20]
)

Jerarquías recuperadas: 1000
Esperadas: 1000
Faltantes: 0
['combined/36217.json', 'combined/63093.json', 'combined/21067.json', 'combined/13332.json', 'combined/70793.json', 'combined/6832.json', 'combined/48282.json', 'combined/56663.json', 'combined/6980.json', 'combined/12027.json', 'combined/40882.json', 'combined/2168.json', 'combined/64788.json', 'combined/23182.json', 'combined/68912.json', 'combined/17455.json', 'combined/9520.json', 'combined/38175.json', 'combined/35569.json', 'combined/31841.json']


In [18]:
sample_ids = list(
    selected_hierarchies.keys()
)[:5]

for rico_id in sample_ids:

    obj = selected_hierarchies[
        rico_id
    ]

    print(
        "\n",
        "=" * 100
    )

    print(
        "RICO ID:",
        rico_id
    )

    print(
        "Tipo:",
        type(obj).__name__
    )

    if isinstance(
        obj,
        dict
    ):
        print(
            "Top-level keys:",
            list(
                obj.keys()
            )
        )

    print(
        json.dumps(
            obj,
            ensure_ascii=False,
            indent=2
        )[:4000]
    )


RICO ID: 36217
Tipo: dict
Top-level keys: ['activity_name', 'activity', 'is_keyboard_deployed', 'request_id']
{
  "activity_name": "mp3.music.musicplayertwoo/com.ijoysoft.music.activity.SettingActivity",
  "activity": {
    "root": {
      "scrollable-horizontal": false,
      "draw": true,
      "ancestors": [
        "android.widget.FrameLayout",
        "android.view.ViewGroup",
        "android.view.View",
        "java.lang.Object"
      ],
      "clickable": false,
      "pressed": "not_pressed",
      "focusable": false,
      "long-clickable": false,
      "enabled": true,
      "bounds": [
        0,
        0,
        1440,
        2392
      ],
      "visibility": "visible",
      "content-desc": [
        null
      ],
      "rel-bounds": [
        0,
        0,
        1440,
        2392
      ],
      "focused": false,
      "selected": false,
      "scrollable-vertical": false,
      "children": [
        {
          "scrollable-horizontal": false,
          "draw": tru

In [19]:
selected_path = (
    INTERIM_DIR
    / "uicrit_rico_selected_hierarchies.jsonl"
)

with open(
    selected_path,
    "w",
    encoding="utf-8"
) as f:

    for rico_id, obj in (
        selected_hierarchies.items()
    ):

        record = {
            "rico_id":
                rico_id,

            "hierarchy":
                obj
        }

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            )
            +
            "\n"
        )

print(
    selected_path.name,
    "✅" if selected_path.exists()
    else "❌"
)

uicrit_rico_selected_hierarchies.jsonl ✅


In [20]:
from collections import Counter

In [21]:
def get_root(obj):

    if not isinstance(obj, dict):
        return None

    activity = obj.get(
        "activity"
    )

    if not isinstance(
        activity,
        dict
    ):
        return None

    return activity.get(
        "root"
    )

In [22]:
def walk_nodes(
    node,
    depth=0
):

    if not isinstance(
        node,
        dict
    ):
        return

    yield node, depth

    children = node.get(
        "children",
        []
    )

    if not isinstance(
        children,
        list
    ):
        return

    for child in children:

        yield from walk_nodes(
            child,
            depth + 1
        )

In [23]:
node_key_counter = Counter()

node_count_total = 0

for rico_id, obj in (
    selected_hierarchies.items()
):

    root = get_root(
        obj
    )

    if root is None:
        continue

    for node, depth in walk_nodes(
        root
    ):

        node_count_total += 1

        node_key_counter.update(
            node.keys()
        )

In [24]:
node_schema = pd.DataFrame(
    node_key_counter.most_common(),
    columns=[
        "node_property",
        "count"
    ]
)

node_schema[
    "coverage_ratio"
] = (
    node_schema["count"]
    /
    node_count_total
)

print(
    "Nodos totales:",
    node_count_total
)

display(
    node_schema.head(50)
)

Nodos totales: 100534


,node_property,count,coverage_ratio
0,scrollable-horizontal,100534,1.000000
1,draw,100534,1.000000
2,ancestors,100534,1.000000
3,clickable,100534,1.000000
4,pressed,100534,1.000000
5,focusable,100534,1.000000
6,long-clickable,100534,1.000000
7,enabled,100534,1.000000
8,bounds,100534,1.000000
9,visibility,100534,1.000000


In [25]:
text_candidate_fields = [
    "text",
    "text-content",
    "textContent",
    "content-desc",
    "resource-id"
]

text_field_audit = []

for field in text_candidate_fields:

    present = 0
    nonempty = 0

    for rico_id, obj in (
        selected_hierarchies.items()
    ):

        root = get_root(
            obj
        )

        if root is None:
            continue

        for node, _ in walk_nodes(
            root
        ):

            if field not in node:
                continue

            present += 1

            value = node.get(
                field
            )

            if isinstance(
                value,
                list
            ):

                valid = any(
                    x is not None
                    and
                    str(x).strip()
                    not in {
                        "",
                        "null",
                        "None"
                    }
                    for x in value
                )

            else:

                valid = (
                    value is not None
                    and
                    str(value).strip()
                    not in {
                        "",
                        "null",
                        "None"
                    }
                )

            if valid:
                nonempty += 1

    text_field_audit.append({
        "field":
            field,

        "present":
            present,

        "nonempty":
            nonempty
    })

text_field_audit = pd.DataFrame(
    text_field_audit
)

text_field_audit

,field,present,nonempty
0,text,29662,23940
1,text-content,0,0
2,textContent,0,0
3,content-desc,100534,2901
4,resource-id,66161,66161


In [26]:
def has_nonempty_value(
    value
):

    if value is None:
        return False

    if isinstance(
        value,
        list
    ):

        return any(
            has_nonempty_value(x)
            for x in value
        )

    text = str(
        value
    ).strip()

    return (
        text != ""
        and
        text.lower()
        not in {
            "none",
            "null",
            "nan"
        }
    )

In [27]:
def is_visible_node(
    node
):

    visible_to_user = node.get(
        "visible-to-user"
    )

    visibility = node.get(
        "visibility"
    )

    if (
        visible_to_user
        is False
    ):
        return False

    if (
        isinstance(
            visibility,
            str
        )
        and
        visibility.lower()
        in {
            "gone",
            "invisible"
        }
    ):
        return False

    return True

In [28]:
def is_interactive_node(
    node
):

    return any([
        node.get(
            "clickable"
        ) is True,

        node.get(
            "focusable"
        ) is True,

        node.get(
            "long-clickable"
        ) is True,

        node.get(
            "scrollable-vertical"
        ) is True,

        node.get(
            "scrollable-horizontal"
        ) is True
    ])

In [29]:
def has_label_proxy(
    node
):

    text_fields = [
        "text",
        "text-content",
        "textContent",
        "content-desc"
    ]

    return any(
        has_nonempty_value(
            node.get(field)
        )
        for field in text_fields
    )

In [30]:
def extract_rico_features(
    rico_id,
    obj
):

    root = get_root(
        obj
    )

    if root is None:

        return {
            "rico_id":
                rico_id,

            "rico_parse_ok":
                False
        }

    nodes = list(
        walk_nodes(
            root
        )
    )

    node_objects = [
        node
        for node, depth
        in nodes
    ]

    depths = [
        depth
        for node, depth
        in nodes
    ]

    visible_nodes = [
        node
        for node
        in node_objects
        if is_visible_node(
            node
        )
    ]

    interactive_nodes = [
        node
        for node
        in visible_nodes
        if is_interactive_node(
            node
        )
    ]

    labeled_interactive = [
        node
        for node
        in interactive_nodes
        if has_label_proxy(
            node
        )
    ]

    leaves = [
        node
        for node
        in node_objects
        if not isinstance(
            node.get(
                "children"
            ),
            list
        )
        or
        len(
            node.get(
                "children",
                []
            )
        )
        == 0
    ]

    internal_nodes = [
        node
        for node
        in node_objects
        if isinstance(
            node.get(
                "children"
            ),
            list
        )
        and
        len(
            node.get(
                "children"
            )
        )
        > 0
    ]

    branching_values = [
        len(
            node.get(
                "children",
                []
            )
        )
        for node
        in internal_nodes
    ]

    resource_id_count = sum(
        has_nonempty_value(
            node.get(
                "resource-id"
            )
        )
        for node
        in visible_nodes
    )

    class_values = [
        str(
            node.get(
                "class"
            )
        )
        for node
        in visible_nodes
        if has_nonempty_value(
            node.get(
                "class"
            )
        )
    ]

    scrollable_count = sum(
        (
            node.get(
                "scrollable-vertical"
            )
            is True
        )
        or
        (
            node.get(
                "scrollable-horizontal"
            )
            is True
        )
        for node
        in visible_nodes
    )

    return {
        "rico_id":
            rico_id,

        "rico_parse_ok":
            True,

        "rico_node_count":
            len(
                node_objects
            ),

        "rico_visible_node_count":
            len(
                visible_nodes
            ),

        "rico_leaf_count":
            len(
                leaves
            ),

        "rico_max_tree_depth":
            (
                max(
                    depths
                )
                if depths
                else 0
            ),

        "rico_mean_branching_factor":
            (
                np.mean(
                    branching_values
                )
                if branching_values
                else 0
            ),

        "rico_interactive_node_count":
            len(
                interactive_nodes
            ),

        "rico_scrollable_node_count":
            scrollable_count,

        "rico_resource_id_ratio":
            (
                resource_id_count
                /
                len(
                    visible_nodes
                )
                if visible_nodes
                else np.nan
            ),

        "rico_interactive_label_proxy_ratio":
            (
                len(
                    labeled_interactive
                )
                /
                len(
                    interactive_nodes
                )
                if interactive_nodes
                else np.nan
            ),

        "rico_unlabeled_interactive_count":
            (
                len(
                    interactive_nodes
                )
                -
                len(
                    labeled_interactive
                )
            ),

        "rico_unique_class_count":
            len(
                set(
                    class_values
                )
            ),

        "rico_class_diversity_ratio":
            (
                len(
                    set(
                        class_values
                    )
                )
                /
                len(
                    class_values
                )
                if class_values
                else np.nan
            )
    }

In [31]:
rico_feature_records = []

for rico_id, obj in (
    selected_hierarchies.items()
):

    rico_feature_records.append(
        extract_rico_features(
            rico_id,
            obj
        )
    )

rico_features = pd.DataFrame(
    rico_feature_records
)

In [32]:
print(
    "Shape:",
    rico_features.shape
)

print(
    "IDs únicos:",
    rico_features[
        "rico_id"
    ].nunique()
)

print(
    "Parse OK:"
)

print(
    rico_features[
        "rico_parse_ok"
    ].value_counts(
        dropna=False
    )
)

display(
    rico_features.head(20)
)

Shape: (1000, 14)
IDs únicos: 1000
Parse OK:
rico_parse_ok
True    1000
Name: count, dtype: int64


,rico_id,rico_parse_ok,rico_node_count,rico_visible_node_count,rico_leaf_count,rico_max_tree_depth,rico_mean_branching_factor,rico_interactive_node_count,rico_scrollable_node_count,rico_resource_id_ratio,rico_interactive_label_proxy_ratio,rico_unlabeled_interactive_count,rico_unique_class_count,rico_class_diversity_ratio
0,36217,True,44,41,21,7,1.869565,2,1,0.560976,0.000000,2,9,0.219512
1,63093,True,124,53,76,15,2.562500,7,0,0.641509,0.428571,4,18,0.339623
2,21067,True,257,172,146,21,2.306306,36,1,0.668605,0.111111,32,24,0.139535
3,13332,True,89,26,51,11,2.315789,10,0,0.500000,0.600000,4,20,0.769231
4,70793,True,46,29,25,9,2.142857,2,1,0.586207,0.500000,1,13,0.448276
5,6832,True,30,17,18,10,2.416667,3,0,0.647059,0.666667,1,11,0.647059
6,48282,True,170,49,91,15,2.139241,12,1,0.612245,0.333333,8,29,0.591837
7,56663,True,23,18,10,6,1.692308,5,1,0.611111,0.600000,2,15,0.833333
8,6980,True,43,28,30,8,3.230769,7,1,0.214286,0.857143,1,10,0.357143
9,12027,True,13,12,8,5,2.400000,0,0,0.750000,NaN,0,6,0.500000


In [33]:
display(
    rico_features.describe(
        include="all"
    ).T
)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
rico_id,1000.0,NaN,NaN,NaN,35824.607,20946.104462,15.0,17136.25,36145.0,53426.5,72194.0
rico_parse_ok,1000,1,True,1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rico_node_count,1000.0,NaN,NaN,NaN,100.534,90.580791,9.0,36.0,70.5,135.25,555.0
rico_visible_node_count,1000.0,NaN,NaN,NaN,47.581,33.875049,7.0,24.0,38.0,60.0,333.0
rico_leaf_count,1000.0,NaN,NaN,NaN,57.793,53.699216,4.0,20.0,40.0,77.0,361.0
rico_max_tree_depth,1000.0,NaN,NaN,NaN,11.686,3.937119,4.0,9.0,11.0,14.0,25.0
rico_mean_branching_factor,1000.0,NaN,NaN,NaN,2.310983,0.508816,1.481481,2.010971,2.227273,2.492564,9.137931
rico_interactive_node_count,1000.0,NaN,NaN,NaN,9.992,8.77536,0.0,4.0,8.0,13.0,80.0
rico_scrollable_node_count,1000.0,NaN,NaN,NaN,0.784,0.756251,0.0,0.0,1.0,1.0,6.0
rico_resource_id_ratio,1000.0,NaN,NaN,NaN,0.590395,0.141507,0.022727,0.51053,0.595238,0.684211,0.9375


In [34]:
numeric_feature_cols = [
    col
    for col
    in rico_features.columns
    if col.startswith(
        "rico_"
    )
    and
    col
    not in {
        "rico_id",
        "rico_parse_ok"
    }
]

for col in numeric_feature_cols:

    print(
        "\n",
        "=" * 80
    )

    print(
        col
    )

    print(
        rico_features[
            col
        ].describe()
    )


rico_node_count
count    1000.000000
mean      100.534000
std        90.580791
min         9.000000
25%        36.000000
50%        70.500000
75%       135.250000
max       555.000000
Name: rico_node_count, dtype: float64

rico_visible_node_count
count    1000.000000
mean       47.581000
std        33.875049
min         7.000000
25%        24.000000
50%        38.000000
75%        60.000000
max       333.000000
Name: rico_visible_node_count, dtype: float64

rico_leaf_count
count    1000.000000
mean       57.793000
std        53.699216
min         4.000000
25%        20.000000
50%        40.000000
75%        77.000000
max       361.000000
Name: rico_leaf_count, dtype: float64

rico_max_tree_depth
count    1000.000000
mean       11.686000
std         3.937119
min         4.000000
25%         9.000000
50%        11.000000
75%        14.000000
max        25.000000
Name: rico_max_tree_depth, dtype: float64

rico_mean_branching_factor
count    1000.000000
mean        2.310983
std         0.

In [35]:
rico_feature_semantics = pd.DataFrame([
    {
        "rico_variable":
            "rico_max_tree_depth",

        "related_construct":
            "STRUCTURAL_COMPLEXITY",

        "possible_heuristic_relation":
            "H6",

        "canonical_equivalent":
            False,

        "must_not_call":
            "navigation_depth",

        "reason":
            "Profundidad del árbol Android no equivale a profundidad de navegación."
    },

    {
        "rico_variable":
            "rico_interactive_label_proxy_ratio",

        "related_construct":
            "CONTROL_RECOGNIZABILITY",

        "possible_heuristic_relation":
            "H6",

        "canonical_equivalent":
            False,

        "must_not_call":
            "accessible_name_ratio",

        "reason":
            "content-desc/text de Android es un proxy, no Accessible Name del DOM web."
    },

    {
        "rico_variable":
            "rico_unlabeled_interactive_count",

        "related_construct":
            "CONTROL_RECOGNIZABILITY",

        "possible_heuristic_relation":
            "H6",

        "canonical_equivalent":
            False,

        "must_not_call":
            "unlabeled_controls_count",

        "reason":
            "La semántica de etiquetado Android y web no es idéntica."
    },

    {
        "rico_variable":
            "rico_class_diversity_ratio",

        "related_construct":
            "STRUCTURAL_CONSISTENCY",

        "possible_heuristic_relation":
            "H4",

        "canonical_equivalent":
            False,

        "must_not_call":
            "component_style_consistency_ratio",

        "reason":
            "Las clases Android no representan estilos CSS."
    },

    {
        "rico_variable":
            "rico_resource_id_ratio",

        "related_construct":
            "STRUCTURAL_IDENTIFIABILITY",

        "possible_heuristic_relation":
            "H4/H6",

        "canonical_equivalent":
            False,

        "must_not_call":
            None,

        "reason":
            "Indicador estructural auxiliar específico de RICO."
    }
])

rico_feature_semantics

,rico_variable,related_construct,possible_heuristic_relation,canonical_equivalent,must_not_call,reason
0,rico_max_tree_depth,STRUCTURAL_COMPLEXITY,H6,False,navigation_depth,Profundidad del árbol Android no equivale a profundidad de navegación.
1,rico_interactive_label_proxy_ratio,CONTROL_RECOGNIZABILITY,H6,False,accessible_name_ratio,"content-desc/text de Android es un proxy, no Accessible Name del DOM web."
2,rico_unlabeled_interactive_count,CONTROL_RECOGNIZABILITY,H6,False,unlabeled_controls_count,La semántica de etiquetado Android y web no es idéntica.
3,rico_class_diversity_ratio,STRUCTURAL_CONSISTENCY,H4,False,component_style_consistency_ratio,Las clases Android no representan estilos CSS.
4,rico_resource_id_ratio,STRUCTURAL_IDENTIFIABILITY,H4/H6,False,NaN,Indicador estructural auxiliar específico de RICO.


In [36]:
ratings_per_ui = (
    uicrit
    .groupby(
        "rico_id_num"
    )
    .size()
)

print(
    ratings_per_ui.value_counts()
)

print(
    "Min:",
    ratings_per_ui.min()
)

print(
    "Max:",
    ratings_per_ui.max()
)

3    966
2     25
4      8
1      1
Name: count, dtype: int64
Min: 1
Max: 4


In [37]:
rating_variability = (
    uicrit
    .groupby(
        "rico_id_num"
    )
    .agg(
        n_records=(
            "rico_id",
            "size"
        ),

        learnability_mean=(
            "learnability",
            "mean"
        ),

        learnability_std=(
            "learnability",
            "std"
        ),

        learnability_min=(
            "learnability",
            "min"
        ),

        learnability_max=(
            "learnability",
            "max"
        ),

        design_quality_mean=(
            "design_quality_rating",
            "mean"
        ),

        design_quality_std=(
            "design_quality_rating",
            "std"
        ),

        design_quality_min=(
            "design_quality_rating",
            "min"
        ),

        design_quality_max=(
            "design_quality_rating",
            "max"
        )
    )
    .reset_index()
)

display(
    rating_variability.head(20)
)

,rico_id_num,n_records,learnability_mean,learnability_std,learnability_min,learnability_max,design_quality_mean,design_quality_std,design_quality_min,design_quality_max
0,15,3,3.333333,0.57735,3.0,4.0,5.333333,1.154701,4,6
1,28,3,2.333333,0.57735,2.0,3.0,5.000000,1.000000,4,6
2,67,3,2.333333,0.57735,2.0,3.0,5.666667,0.577350,5,6
3,190,3,3.333333,0.57735,3.0,4.0,6.666667,0.577350,6,7
4,193,3,3.000000,1.00000,2.0,4.0,5.000000,2.000000,3,7
5,233,3,2.333333,0.57735,2.0,3.0,4.666667,0.577350,4,5
6,288,3,2.333333,0.57735,2.0,3.0,5.000000,1.000000,4,6
7,342,3,2.333333,0.57735,2.0,3.0,5.333333,1.527525,4,7
8,422,3,2.666667,0.57735,2.0,3.0,6.000000,0.000000,6,6
9,445,3,3.000000,1.00000,2.0,4.0,6.666667,0.577350,6,7


In [38]:
display(
    rating_variability[
        [
            "n_records",
            "learnability_std",
            "design_quality_std"
        ]
    ]
    .describe()
)

,n_records,learnability_std,design_quality_std
count,1000.000000,999.000000,999.000000
mean,2.981000,0.509896,0.857508
std,0.191509,0.341075,0.528495
min,1.000000,0.000000,0.000000
25%,3.000000,0.000000,0.577350
50%,3.000000,0.577350,0.577350
75%,3.000000,0.577350,1.154701
max,4.000000,1.527525,2.645751


In [39]:
uicrit_proxy_targets = (
    uicrit
    .groupby(
        "rico_id_num"
    )
    .agg(
        proxy_record_count=(
            "rico_id",
            "size"
        ),

        learnability_proxy=(
            "learnability",
            "median"
        ),

        design_quality_proxy=(
            "design_quality_rating",
            "median"
        ),

        learnability_proxy_mean=(
            "learnability",
            "mean"
        ),

        design_quality_proxy_mean=(
            "design_quality_rating",
            "mean"
        )
    )
    .reset_index()
    .rename(
        columns={
            "rico_id_num":
                "rico_id"
        }
    )
)

In [40]:
print(
    uicrit_proxy_targets.shape
)

print(
    "IDs únicos:",
    uicrit_proxy_targets[
        "rico_id"
    ].nunique()
)

display(
    uicrit_proxy_targets.head(20)
)

(1000, 6)
IDs únicos: 1000


,rico_id,proxy_record_count,learnability_proxy,design_quality_proxy,learnability_proxy_mean,design_quality_proxy_mean
0,15,3,3.0,6.0,3.333333,5.333333
1,28,3,2.0,5.0,2.333333,5.000000
2,67,3,2.0,6.0,2.333333,5.666667
3,190,3,3.0,7.0,3.333333,6.666667
4,193,3,3.0,5.0,3.000000,5.000000
5,233,3,2.0,5.0,2.333333,4.666667
6,288,3,2.0,5.0,2.333333,5.000000
7,342,3,2.0,5.0,2.333333,5.333333
8,422,3,3.0,6.0,2.666667,6.000000
9,445,3,3.0,7.0,3.000000,6.666667


In [41]:
uicrit_rico_proxy = (
    rico_features
    .merge(
        uicrit_proxy_targets,
        on="rico_id",
        how="inner",
        validate="one_to_one"
    )
)

In [42]:
print(
    "Dataset proxy:",
    uicrit_rico_proxy.shape
)

print(
    "IDs únicos:",
    uicrit_rico_proxy[
        "rico_id"
    ].nunique()
)

print(
    "Duplicados:",
    uicrit_rico_proxy[
        "rico_id"
    ].duplicated()
    .sum()
)

Dataset proxy: (1000, 19)
IDs únicos: 1000
Duplicados: 0


In [43]:
uicrit_rico_proxy[
    "source_dataset"
] = "UICrit_RICO"

uicrit_rico_proxy[
    "label_level"
] = "RATING_PROXY"

uicrit_rico_proxy[
    "primary_training"
] = False

uicrit_rico_proxy[
    "feature_modality"
] = "ANDROID_UI_HIERARCHY"

In [44]:
rico_features.to_csv(
    INTERIM_DIR
    / "rico_structural_features.csv",
    index=False
)

uicrit_proxy_targets.to_csv(
    INTERIM_DIR
    / "uicrit_proxy_targets.csv",
    index=False
)

uicrit_rico_proxy.to_csv(
    INTERIM_DIR
    / "uicrit_rico_proxy_dataset.csv",
    index=False
)

node_schema.to_csv(
    REPORTS_DIR
    / "rico_node_schema.csv",
    index=False
)

text_field_audit.to_csv(
    REPORTS_DIR
    / "rico_text_field_audit.csv",
    index=False
)

rico_feature_semantics.to_csv(
    REPORTS_DIR
    / "rico_feature_semantics.csv",
    index=False
)

rating_variability.to_csv(
    REPORTS_DIR
    / "uicrit_rating_variability.csv",
    index=False
)

In [45]:
expected = [
    INTERIM_DIR
    / "rico_structural_features.csv",

    INTERIM_DIR
    / "uicrit_proxy_targets.csv",

    INTERIM_DIR
    / "uicrit_rico_proxy_dataset.csv",

    REPORTS_DIR
    / "rico_node_schema.csv",

    REPORTS_DIR
    / "rico_text_field_audit.csv",

    REPORTS_DIR
    / "rico_feature_semantics.csv",

    REPORTS_DIR
    / "uicrit_rating_variability.csv"
]

for path in expected:

    print(
        path.name,
        "✅" if path.exists()
        else "❌"
    )

rico_structural_features.csv ✅
uicrit_proxy_targets.csv ✅
uicrit_rico_proxy_dataset.csv ✅
rico_node_schema.csv ✅
rico_text_field_audit.csv ✅
rico_feature_semantics.csv ✅
uicrit_rating_variability.csv ✅
